# Earthworks
This notebook runs the examples in the [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html) manual page. See the [tutorials](https://grass-tutorials.osgeo.org/content/tutorials/earthworks/earthworks.html) page for more examples.

## Setup
Start a GRASS session in the [North Carolina Basic Dataset](https://grass.osgeo.org/sampledata/north_carolina/nc_basic_spm_grass7.zip).

In [ ]:
# Import modules
import sys
import subprocess

# Find GRASS Python packages
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

# Import GRASS packages
import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

# Start GRASS session
session = gj.init("~/grassdata/nc_basic_spm_grass7/user1")
tools = Tools(session=session, overwrite=True)

## Installation

Install the addon with [g.extension](https://grass.osgeo.org/grass-stable/manuals/g.extension.html).

In [ ]:
# Install addon
tools.g_extension(extension="r.earthworks")

## Basic Operations

Set the computational region with [g.region](https://grass.osgeo.org/grass-stable/manuals/g.region.html) and use map algebra to generate a flat terrain with [r.mapcalc](https://grass.osgeo.org/grass-stable/manuals/r.mapcalc.html).

In [ ]:
# Set region
tools.g_region(n=200, e=800, s=0, w=0, res=1)

# Generate elevation
gs.mapcalc("terrain = 0")

# Set color gradient
tools.r_colors(map="terrain", color="viridis")

# Visualize
m = gj.Map(width=1200)
m.d_rast(map="terrain")
m.show()

### Fill Operation
Model a peak from a set of x- and y-coordinates with [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html). Use the z parameter to set a z-coordinate for the top of the peak. Optionally use the flat parameter to create a plateau at the top of the peak. 

In [ ]:
# Model earthworks
tools.r_earthworks(
    elevation="terrain",
    earthworks="peak",
    operation="fill",
    function="linear",
    linear=0.5,
    coordinates=[400, 100],
    z=25,
    flat=25,
)

# Visualize
m = gj.Map(width=1600)
m.d_rast(map="peak")
m.d_legend(raster="peak", color="white", at=(5, 95, 1, 3))
m.save("r_earthworks_01.png")
m.show()

### Cut Operation
Model a pit from a set of x- and y-coordinates with [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html). Set a z-coordinate for the bottom of the pit.

In [ ]:
# Model earthworks
tools.r_earthworks(
    elevation="terrain",
    earthworks="pit",
    operation="cut",
    function="linear",
    linear=0.5,
    coordinates=[400, 100],
    z=-25,
    flat=25,
)

# Visualize
m = gj.Map(width=1600)
m.d_rast(map="pit")
m.d_legend(raster="pit", color="black", at=(5, 95, 1, 3))
m.save("r_earthworks_02.png")
m.show()

### Cut & Fill Operation
Model a pit and a peak from two sets of x- and y-coordinates with [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html). Set a z-coordinate for the bottom of the pit and another z-coordinate for the top of the peak.

In [ ]:
# Model earthworks
tools.r_earthworks(
    elevation="terrain",
    earthworks="pit_and_peak",
    operation="cutfill",
    function="linear",
    linear=0.5,
    coordinates=[350, 100, 450, 100],
    z=[-25, 25],
    flat=25,
)

# Visualize
m = gj.Map(width=1600)
m.d_rast(map="pit_and_peak")
m.d_legend(raster="pit_and_peak", color="black", at=(5, 95, 1, 3))
m.save("r_earthworks_03.png")
m.show()

### Random Earthworks

Model a set of random peaks by generating a random surface, sampling random points, and then filling on those points. First generate a random surface with [r.surf.random](https://grass.osgeo.org/grass-stable/manuals/r.surf.random.html). Set a seed for reproducible results or set flags="s" for a random seed. Then randomly sample 50 cells from the random surface [r.random](https://grass.osgeo.org/grass-stable/manuals/r.random.html). Use a fill operation to model random peaks with [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html). Set the input raster to the random cells.

In [ ]:
# Generate random points
tools.r_surf_random(out="surface", min=0, max=25, seed=2)
tools.r_random(input="surface", npoints=50, raster="random", seed=7)

# Model earthworks
tools.r_earthworks(
    elevation="terrain",
    earthworks="random_earthworks",
    operation="fill",
    raster="random",
    function="linear",
    linear=0.25,
    flat=25,
)

# Visualize
m = gj.Map(width=1600)
m.d_rast(map="random_earthworks")
m.d_legend(raster="random_earthworks", digits=0, at=(5, 95, 1, 3))
m.save("r_earthworks_04.png")
m.show()

Compute contours for earthworks with [r.contour](https://grass.osgeo.org/grass-stable/manuals/r.contour.html).

In [ ]:
# Derive contours
tools.r_contour(input="random_earthworks", output="contours", step=2)

# Set hypsometric color gradient
tools.v_colors(map="contours", use="attr", column="level", color="grey", flags="n")

# Visualize Contours
m = gj.Map(width=1600)
m.d_vect(map="contours")
m.save("r_earthworks_05.png")
m.show()

## Road Grading
Use a vector map of a road network to grade a road crossing over a valley. Start GRASS in the North Carolina basic dataset. First set the computation region with [g.region](https://grass.osgeo.org/grass-stable/manuals/g.region.html). Then run [r.earthworks](https://grass.osgeo.org/grass-stable/manuals/addons/r.earthworks.html) with the input elevation set to `elevation`, input lines set to `roadsmajor`, z set to `95`, operation set to `fill`, function set to `linear`, linear set to `0.25`, and flat set to `25`. Use flag `-p` to print the volume of fill. This will grade an embankment through the valley with a 50 meter wide roadway at a constant elevation of 95 meters with side slopes of 25 percent.

### Existing Conditions

Visualize the existing topography in the study area.

In [ ]:
# Set region
tools.g_region(n=217150, s=216550, w=638750, e=641150, res=10)

# Extract region
gs.mapcalc("site = elevation")

# Set color gradient
tools.r_colors(map="site", color="viridis")

# Visualize existing conditions
m = gj.Map(width=1600)
m.d_rast(map="site")
m.d_legend(raster="site", digits=0, at=(5, 95, 1, 3))
m.save("r_earthworks_06.png")
m.show()

### Grade Embankment

Construct an embankment for the roadway.

In [ ]:
# Grade embankment
earthwork = tools.r_earthworks(
    elevation="elevation",
    earthworks="embankment",
    lines="roadsmajor",
    volume="volume",
    z=95,
    function="linear",
    linear=0.25,
    operation="fill",
    flat=25,
    flags="p",
)

# Visualize proposed design
m = gj.Map(width=1600)
m.d_rast(map="embankment")
m.d_legend(raster="embankment", digits=0, at=(5, 95, 1, 3))
m.save("r_earthworks_07.png")
m.show()

Visualize the volume of fill.

In [ ]:
# Visualize elevation change
m = gj.Map(width=1600)
m.d_rast(map="volume")
m.d_legend(raster="volume", digits=0, at=(5, 95, 1, 3), color="white")
m.save("r_earthworks_08.png")
m.show()

## Dam Breach

Model a flood due to a dam breach. Use r.earthworks to breach the dam and then use [r.lake](https://grass.osgeo.org/grass-stable/manuals/r.lake.html) to model flood risk. Optionally use [r.relief](https://grass.osgeo.org/grass-stable/manuals/r.relief.html) and [r.skyview](https://grass.osgeo.org/grass-stable/manuals/addons/r.skyview.html) for terrain visualization. 

In [ ]:
# Install addon
tools.g_extension(extension="r.skyview")

### Visualize Dam

In [ ]:
# Set region
tools.g_region(n=223740, s=222740, w=632950, e=636950, res=10)

# Fill lake
tools.r_lake(
    elevation="elevation",
    water_level=104,
    lake="lake",
    coordinates=[635150.7489931877, 223203.9595016748],
)

# Model shaded relief
tools.r_relief(input="elevation", output="relief", zscale=2)
tools.r_grow(input="relief", output="relief")
tools.r_skyview(input="elevation", output="skyview")

# Visualize existing conditions
m = gj.Map(width=1600)
m.d_shade(shade="relief", color="skyview", brighten=50)
m.d_rast(map="lake")
m.d_legend(raster="lake", digits=1, at=(5, 95, 1, 3))
m.save("r_earthworks_09.png")
m.show()

### Model Dam Breach

In [ ]:
# Model breach
tools.r_earthworks(
    elevation="elevation",
    earthworks="breach",
    operation="cut",
    coordinates=[635235.4648198467, 223210.9879314204],
    z=103,
    function="linear",
    linear=0.5,
    flat=20,
)

# Model flood
tools.r_lake(
    elevation="breach",
    water_level=104,
    lake="flood",
    coordinates=[635150.7489931877, 223203.9595016748],
)

# Model shaded relief
tools.r_relief(input="breach", output="relief", zscale=2)
tools.r_grow(input="relief", output="relief")
tools.r_skyview(input="breach", output="skyview")

# Visualize dam breach
m = gj.Map(width=1600)
m.d_shade(shade="relief", color="skyview", brighten=50)
m.d_rast(map="flood")
m.d_legend(raster="flood", digits=1, at=(5, 95, 1, 3))
m.save("r_earthworks_10.png")
m.show()